In [7]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import csv
from datetime import datetime, timezone, timedelta
from openlocationcode import openlocationcode as olc
import os
import json
import random
import numpy as np
from ast import literal_eval

In [ ]:
DATAFOLD = 'NYC' 
stage = "meta"

In [ ]:
def get_pluscode(latitude, longitude):
    if pd.isna(latitude) or pd.isna(longitude):
        return "INVALID"
    try:
        code = olc.encode(float(latitude), float(longitude))
        return code[:6]
    except:
        return "INVALID"



def format_time(row):
    time_str = row['time']
    parts = time_str.split()
    clean_time_str = f"{parts[1]} {parts[2]} {parts[3]} {parts[5]}"
    naive_dt = datetime.strptime(clean_time_str, "%b %d %H:%M:%S %Y")
    
    tz = timezone(timedelta(minutes=int(row['tz_offset'])))
    localized_dt = naive_dt.replace(tzinfo=tz)
    return localized_dt.strftime("%Y-%m-%d %H:%M")

def save_mapping(mapping, file_path):
    with open(file_path, "w", newline="") as uidfile:
        writer = csv.writer(uidfile)
        writer.writerow(["original_uid", "new_uid"])
        for original_uid, new_uid in mapping.items():
            writer.writerow([original_uid, new_uid])


def filter_data(datafold=None, min_user_interactions=10, min_poi_interactions=10):

    os.makedirs(f"{datafold}", exist_ok=True)

    input_file = f"{datafold}.txt"
    df = pd.read_csv(input_file, delimiter="\t", header=None,
                     names=['uid', 'pid', '_', 'category', 'latitude', 'longitude', 'tz_offset', 'time'])

    df['PoiFreq'] = df.groupby('pid')['uid'].transform('count')
    df = df[df['PoiFreq'] >= min_poi_interactions]
    df['UserFreq'] = df.groupby('uid')['pid'].transform('count')
    df = df[df['UserFreq'] >= min_user_interactions]

    df = df.drop(columns=['PoiFreq', 'UserFreq'])

    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    df = df.dropna(subset=['latitude', 'longitude'])

    df["region"] = df.apply(lambda row: get_pluscode(row['latitude'], row['longitude']), axis=1)
    df['formatted_time'] = df.apply(format_time, axis=1)
    
    uids = list(df["uid"].unique())
    pids = list(df["pid"].unique())
    cats = list(df["category"].unique())
    regs = list(df["region"].unique())
    random.shuffle(uids)
    random.shuffle(pids)
    random.shuffle(cats)
    random.shuffle(regs)
    uid_map = {uid: i for i, uid in enumerate(uids)}
    pid_map = {pid: i for i, pid in enumerate(pids)}
    cat_map = {cat: i for i, cat in enumerate(cats)}
    reg_map = {reg: i for i, reg in enumerate(regs)}
    df["new_uid"] = df["uid"].map(uid_map)
    df["new_pid"] = df["pid"].map(pid_map)
    df["new_cid"] = df["category"].map(cat_map)
    df["new_region"] = df["region"].map(reg_map)

    output_file = f"{datafold}/{datafold}.csv"
    df[['new_uid', 'new_pid', 'new_cid', 'category', 'new_region', 'latitude', 'longitude', 'formatted_time']].to_csv(
        output_file, index=False, header=['uid', 'pid', 'cid', 'category', 'region', 'latitude', 'longitude', 'time']
    )

    if not os.path.exists(f"{datafold}/{stage}"):
        os.makedirs(f"{datafold}/{stage}")

    save_mapping(uid_map, f"{datafold}/{stage}/uidmap.csv")
    save_mapping(pid_map, f"{datafold}/{stage}/pidmap.csv")
    save_mapping(cat_map, f"{datafold}/{stage}/cidmap.csv")

    print("处理完成！")

datafold = DATAFOLD
filter_data(datafold, min_user_interactions=10, min_poi_interactions=10)

处理完成！


In [ ]:

def poi_info(datafold):
    file_path = f'{datafold}/{datafold}.csv'
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"错误：{file_path} 文件未找到。")

    df = pd.read_csv(file_path)

    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])

    poi_info_data = []

    for pid, group in df.groupby('pid'):
        row0 = group.iloc[0]
        category = row0['category']
        region = row0['region']
        latitude = row0['latitude']
        longitude = row0['longitude']

        hours = group['time'].dt.hour
        hour_counts = hours.value_counts().to_dict()  # {hour: count}

        count_threshold = 1
        filtered_hour_counts = {int(h): int(c) for h, c in hour_counts.items() if c >= count_threshold}

        sorted_hour_counts = dict(
            sorted(filtered_hour_counts.items(), key=lambda x: x[1], reverse=True)
        )

        poi_info_data.append({
            'pid': pid,
            'category': category,
            'region': region,
            'latitude': latitude,
            'longitude': longitude,
            'visit_time_and_count': sorted_hour_counts
        })

    poi_info_df = pd.DataFrame(poi_info_data)
    output_path = f'{datafold}/poi_info.csv'
    poi_info_df.to_csv(output_path, index=False)

    print(f"成功创建 {output_path}，共 {len(poi_info_df)} 个 POI")

datafold = DATAFOLD
poi_info(datafold)

成功创建 CA/poi_info.csv，共 14027 个 POI


In [ ]:
# 数据切分
def split_data(datafold, train_ratio=0.8, valid_ratio=0.1, test_ratio=0.1):
    file_name = f"{datafold}/{datafold}.csv"
    df = pd.read_csv(file_name)
    df = df[['uid', 'pid', 'time']]
    df = df.sort_values(by='time')
    train_size = int(train_ratio * len(df))
    valid_size = int(valid_ratio * len(df))
    
    train_df = df[:train_size]
    valid_df = df[train_size:train_size + valid_size]
    test_df = df[train_size + valid_size:]

    def remove_users_pois_test(df_train, df_test):
        users_train = df_train['uid'].unique()
        pois_train = df_train['pid'].unique()
        df_test = df_test[df_test['uid'].isin(users_train)]
        df_test = df_test[df_test['pid'].isin(pois_train)]
        return df_test

    train_df.to_csv(f'{datafold}/train_data.csv', index=False)

    valid_df = remove_users_pois_test(train_df, valid_df)
    valid_uids = valid_df['uid'].unique()
    expanded_valid_df = df[df['uid'].isin(valid_uids)]
    expanded_valid_df.to_csv(f'{datafold}/valid_data.csv', index=False)

    test_df = remove_users_pois_test(train_df, test_df)
    test_uids = test_df['uid'].unique()
    expanded_test_df = df[df['uid'].isin(test_uids)]
    expanded_test_df.to_csv(f'{datafold}/test_data.csv', index=False)

datafold = DATAFOLD
split_data(datafold, train_ratio=0.8, valid_ratio=0.1)
